In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# <span style='color:gray'>Example of explanations plots for ALL model</span> 

## General elements

### Training model $f_\theta$

In [ ]:
# Basic libraries
import numpy as np
import pickle as pk
import time_explainer as time
from tensorflow.keras.models import load_model

In [ ]:
import sys
import tensorflow as tf
import keras

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)


If Keras is 3.X, execute the next cell and restart the kernel. If Keras is 2.X, skip the next cell and continue with the notebook.

In [ ]:
import sys
!{sys.executable} -m pip uninstall -y tensorflow keras
!{sys.executable} -m pip install tensorflow==2.15.1 keras==2.15.0

In [ ]:
# Download the model
path = 'data/calendar_weather_holiday_in_advance/'  
model_name = 'model_calendar_wather_holiday_in_advance.h5'
model = load_model(path + model_name, compile=False)
model.summary()

### Uniform random sample of $X_{train}$ associated with the fitting of $\theta$

In [ ]:
## Get sample (100 random samples) of the training set to use in the notebook
with open(path + 'X_train_sample.pkl', 'rb') as f:
    X_train_sample = pk.load(f)

### Initial interpretable decomposition of  $X$ in the domain of $f_\theta$

In [ ]:
# Get some X, e.g. the first element of the test set
with open(path + 'X_encoder_test.pkl', 'rb') as f:
    X_encoder_test = pk.load(f)

with open(path + 'X_decoder_test.pkl', 'rb') as f:
    X_decoder_test = pk.load(f)

X_observation = [X_encoder_test[0], X_decoder_test[0]]

## Plots

### Initial exploration

In [ ]:
##### ALL model #####
### Initial decomposition configuration
component_names = ['past-load','calendar','weather','holiday']

def ALL_initial(X):
    X_split = {0: X[0][:,0:1], # past-load
               1: [X[0][:,1:4],X[1][:,0:3]], # calendar
               2: [X[0][:,4:5],X[1][:,3:4]], # weather
               3: [X[0][:,5:15],X[1][:,4:14]]# holiday
                }         
    return X_split

def joint_ALL_initial(X_split):
    X_encoder = np.concatenate([X_split[0],X_split[1][0],X_split[2][0],X_split[3][0]], axis = 1).reshape(180, 15)
    X_decoder = np.concatenate([X_split[1][1],X_split[2][1],X_split[3][1]], axis = 1).reshape(36, 14)
    return [X_encoder,X_decoder]

In [ ]:
ALL_initial = time.elements(model, X_train_sample, split_func=ALL_initial, joint_func=joint_ALL_initial)

In [ ]:
initial_explainability_model = ALL_initial.explainability_model(X_observation)
fig = time.local_plot(initial_explainability_model,component_names)
fig.show()

### Zooms

In [ ]:
##### ALL model #####
### Zoom over past-load
component_names = ['past-load 72-180 h before','past-load 24-72 h before','past-load 1-24 h before']

def past_load_zoom(X):
    X_split = {0: X[0][:-72,0:1], # past-load 72-180 h before
               1: X[0][-72:-24,0:1], # past-load 24-72 h before
               2: X[0][-24:,0:1], # past-load 1-24 h before
               'rest_calendar':[X[0][:,1:4],X[1][:,0:3]], # fixed rest
               'rest_weather': [X[0][:,4:5],X[1][:,3:4]], # fixed rest
               'rest_holiday': [X[0][:,5:15],X[1][:,4:14]] # fixed rest 
                }         
    return X_split

def join_past_load_zoom(X_split):
    X_past_load = np.concatenate([X_split[0],X_split[1],X_split[2]], axis = 0).reshape(180, 1)
    X_encoder = np.concatenate([X_past_load,X_split['rest_calendar'][0], X_split['rest_weather'][0], X_split['rest_holiday'][0]], axis = 1).reshape(180, 15)
    X_decoder = np.concatenate([X_split['rest_calendar'][1],X_split['rest_weather'][1],X_split['rest_holiday'][1]], axis = 1).reshape(36, 14)
    return [X_encoder,X_decoder]

In [ ]:
phi_zero = initial_explainability_model[0]
phi_close_up = initial_explainability_model[1][0]
ALL_past_load_zoom = time.elements(model, 
                                   X_train_sample, 
                                   component_to_close_up=phi_close_up, 
                                   initial_bias=phi_zero, 
                                   split_func=past_load_zoom, 
                                   joint_func=join_past_load_zoom
                                  )

In [ ]:
past_load_zoom_model = ALL_past_load_zoom.close_up_explainability_model(X_observation)
fig = time.local_plot(past_load_zoom_model,component_names)
fig.show()

In [ ]:
##### ALL model #####
### Zoom over past-load past-load 1-24 h before
component_names = ['past-load 12-24 h before','past-load 1-12 h before']

def past_load_1_24_zoom(X):
    X_split = {'rest_past_load': X[0][:-24,0:1], # fixed rest
               0: X[0][-24:-12,0:1], # past-load 24-12 h before 
               1: X[0][-12:,0:1], # past-load 1-12 h before
               'rest_calendar':[X[0][:,1:4],X[1][:,0:3]], # fixed rest
               'rest_weather': [X[0][:,4:5],X[1][:,3:4]], # fixed rest
               'rest_holiday': [X[0][:,5:15],X[1][:,4:14]] # fixed rest 
                }         
    return X_split

def join_past_load_1_24_zoom(X_split):
    X_past_load = np.concatenate([X_split['rest_past_load'],X_split[0],X_split[1]], axis = 0).reshape(180, 1)
    X_encoder = np.concatenate([X_past_load,X_split['rest_calendar'][0], X_split['rest_weather'][0], X_split['rest_holiday'][0]], axis = 1).reshape(180, 15)
    X_decoder = np.concatenate([X_split['rest_calendar'][1],X_split['rest_weather'][1],X_split['rest_holiday'][1]], axis = 1).reshape(36, 14)
    return [X_encoder,X_decoder]

In [ ]:
phi_zero = initial_explainability_model[0]
phi_close_up = past_load_zoom_model[1][2]
ALL_past_load_1_24_zoom = time.elements(model, 
                                        X_train_sample,
                                        component_to_close_up=phi_close_up,
                                        initial_bias=phi_zero,
                                        split_func=past_load_1_24_zoom,
                                        joint_func=join_past_load_1_24_zoom
                                       )

In [ ]:
past_load_1_24_zoom_model = ALL_past_load_1_24_zoom.close_up_explainability_model(X_observation)
base_vector = past_load_zoom_model[1][0] + past_load_zoom_model[1][1] ### for consistent ploting efects, past-load 1-24 h before comes after the rest components
fig = time.local_plot(past_load_1_24_zoom_model,component_names,base_vector)
fig.show()

In [ ]:
##### ALL model #####
### Zoom over calendar
component_names = ['past-calendar 24-180 h before','past-calendar 1-24 h before','future-calendar']

def calendar_zoom(X):
    X_split = {'rest_load': X[0][:,0:1], # fixed rest
               0: X[0][:-24,1:4], #1: ‘past-calendar 24-180 h before’
               1: X[0][-24:,1:4], #2: ‘past-calendar 1-24 h before’
               2: X[1][:,0:3], #3: ‘future-calendar’
               'rest_weather': [X[0][:,4:5],X[1][:,3:4]], # fixed rest
               'rest_holiday': [X[0][:,5:15],X[1][:,4:14]] # fixed rest 
                }         
    return X_split

def joint_calendar_zoom(X_split):
    X_past_calendar = np.concatenate([X_split[0],X_split[1]], axis = 0).reshape(180, 3)
    X_encoder = np.concatenate([X_split['rest_load'],X_past_calendar,X_split['rest_weather'][0],X_split['rest_holiday'][0]], axis = 1).reshape(180, 15)
    X_decoder = np.concatenate([X_split[2],X_split['rest_weather'][1],X_split['rest_holiday'][1]], axis = 1).reshape(36, 14)
    return [X_encoder,X_decoder]

In [ ]:
phi_zero = initial_explainability_model[0]
phi_close_up = initial_explainability_model[1][1]
ALL_calendar_zoom = time.elements(model, 
                                    X_train_sample,
                                    component_to_close_up=phi_close_up,
                                    initial_bias=phi_zero,
                                    split_func=calendar_zoom,
                                    joint_func=joint_calendar_zoom
                                   )

In [ ]:
calendar_zoom_model = ALL_calendar_zoom.close_up_explainability_model(X_observation)
base_vector = initial_explainability_model[1][0]  ### for consistent ploting efects, past-load is below calendar in initial plot
fig = time.local_plot(calendar_zoom_model,component_names,base_vector)
fig.show()

In [ ]:
##### ALL model #####
### Zoom  over weather
component_names = ['past-weather','future-weather']

def weather_zoom(X):
    X_split = {'rest_load': X[0][:,0:1], # fixed rest
               'rest_calendar': [X[0][:,1:4],X[1][:,0:3]], # fixed_rest
               0: X[0][:,4:5], # past-weather
               1: X[1][:,3:4], # future_weather
               'rest_holiday': [X[0][:,5:15],X[1][:,4:14]] # fixed rest 
                }         
    return X_split

def joint_weather_zoom(X_split):
    X_encoder = np.concatenate([X_split['rest_load'],X_split['rest_calendar'][0],X_split[0],X_split['rest_holiday'][0]], axis = 1).reshape(180, 15)
    X_decoder = np.concatenate([X_split['rest_calendar'][1],X_split[1],X_split['rest_holiday'][1]], axis = 1).reshape(36, 14)
    return [X_encoder,X_decoder]

In [ ]:
phi_zero = initial_explainability_model[0]
phi_close_up = initial_explainability_model[1][2] 
ALL_weather_zoom = time.elements(model, 
                                 X_train_sample,
                                 component_to_close_up=phi_close_up,
                                 initial_bias=phi_zero,
                                 split_func=weather_zoom, 
                                 joint_func=joint_weather_zoom
                                )

In [ ]:
weather_zoom_model = ALL_weather_zoom.close_up_explainability_model(X_observation)
base_vector = initial_explainability_model[1][0] + initial_explainability_model[1][1]  ### for consistent ploting efects, calendar is below weather in initial plot
fig = time.local_plot(weather_zoom_model,component_names,base_vector)
fig.show()

In [ ]:
##### ALL model #####
### Zoom over holiday
component_names = ['past-holiday','future-holiday']

def holiday_zoom(X):
    X_split = {'rest_load': X[0][:,0:1], # fixed rest
               'rest_calendar': [X[0][:,1:4],X[1][:,0:3]], # fixed rest
               'rest_weather': [X[0][:,4:5],X[1][:,3:4]], # fixed rest
               0: X[0][:,5:15], # past-holiday
               1: X[1][:,4:14] # future-holiday 
                }         
    return X_split

def joint_holiday_zoom(X_split):
    X_encoder = np.concatenate([X_split['rest_load'],X_split['rest_calendar'][0],X_split['rest_weather'][0],X_split[0]], axis = 1).reshape(180, 15)
    X_decoder = np.concatenate([X_split['rest_calendar'][1],X_split['rest_weather'][1],X_split[1]], axis = 1).reshape(36, 14)
    return [X_encoder,X_decoder]

In [ ]:
phi_zero = initial_explainability_model[0]
phi_close_up = initial_explainability_model[1][3] 
ALL_holiday_zoom = time.elements(model, 
                                 X_train_sample,
                                 component_to_close_up=phi_close_up,
                                 initial_bias=phi_zero, 
                                 split_func=holiday_zoom, 
                                 joint_func=joint_holiday_zoom
                                )

In [ ]:
holiday_zoom_model = ALL_holiday_zoom.close_up_explainability_model(X_observation)
base_vector = initial_explainability_model[1][0] + initial_explainability_model[1][1] + initial_explainability_model[1][2]  ### for consistent ploting efects, calendar is below weather in initial plot
fig = time.local_plot(holiday_zoom_model,component_names,base_vector)
fig.show()